Plot line counts for a single repository over time. Collect the data first with:

    code-metrics repo-history https://github.com/lsst/daf_butler

In [ ]:
%matplotlib widget

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from lsst.codemetrics.plotting import (
    PYTHON_ALIASES,
    apply_aliases,
    c_family_aliases,
    insert_gaps,
    load_repo,
    pivot,
    select,
    top_series,
)

In [ ]:
name = "daf_butler"
frame = load_repo(name)
frame.head()

In [ ]:
# select() with no counter= uses the file's sole backend and raises if
# it holds several. Pass counter="cloc" to choose one explicitly.
chosen = select(frame)

# Gather each language's whole size into one series. Headers join the
# source they belong to, under whichever of C and C++ this repository
# actually holds, and notebooks join Python. Both maps cover all three
# backends; unused keys are simply no-ops.
#
# SWIG stays on its own. It is neither the C++ it wraps nor the Python
# it presents, and it is large enough in a pre-pybind11 history that
# putting it in either would misstate both.
folded = apply_aliases(chosen, c_family_aliases(chosen) | PYTHON_ALIASES)

# Any real repository reports enough languages to bury the plot under
# its own legend. Rank language-and-measure pairs by the largest each
# ever reached, so a subsystem that grew and was later removed still
# shows its rise and fall, and a series that is flat zero (JSON has no
# comments) never takes a slot from one that carries content.
series = top_series(folded, n=10)
series

In [ ]:
# Counts only change where a revision was measured, so draw them as
# steps rather than sloping between measurements.
#
# A step carried across a quiet stretch is true -- no commits means the
# counts did not change -- but it reads as a measured plateau. insert_gaps
# blanks the line where nothing was committed for longer than max_gap, so
# a quiet stretch looks quiet. Raise max_gap for a repository that is
# worked on in occasional bursts, or drop the call to draw straight
# through. A blank stretch means no development, not missing data.
max_gap = "90D"
wide = {measure: insert_gaps(pivot(folded, value=measure), max_gap=max_gap) for _, measure in series}

fig, ax = plt.subplots()
for language, measure in series:
    line = wide[measure][language]
    ax.plot(line.index, line, label=f"{language} {measure}", drawstyle="steps-post")
ax.set_title(f"Lines of {name} code and comments")
ax.set_ylim(bottom=0)

# Let the tick labels follow the span. A repository covering a few
# months would otherwise get nine "2025-12"-style labels that collide,
# where one covering years gets bare years that fit comfortably.
locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

# Park the legend outside the axes so it can never cover the curves.
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0, fontsize="small")

In [ ]:
# Uncomment to save a PDF for publications.
# ax.set_title("")
# fig.savefig(f"{name}-lines.pdf", bbox_inches="tight")

## Code and comments together

The plot above draws code and comment as separate series, which shows how a language is written.
This one sums them, which shows how much of it there is, and is usually the figure a paper wants.

In [ ]:
# c_family_aliases decided from the counts whether this repository's C
# family is named C, C++, or kept apart as both alongside cloc's shared
# headers, so read the names back from the folded frame rather than
# assuming one of them. A language that is flat zero throughout is left
# out: it would draw along the axis and spend a legend entry on nothing.
peaks = folded.groupby("language")["lines"].max()
languages = [language for language in ("Python", "C", "C++", "C/C++ Header") if peaks.get(language, 0) > 0]

# Steps and gaps for the same reasons as above, and from the same
# revisions, so the breaks fall on the same dates in both figures.
combined = insert_gaps(pivot(select(folded, languages=languages), value="lines"), max_gap=max_gap)

fig, ax = plt.subplots()
for language in languages:
    ax.plot(combined.index, combined[language], label=language, drawstyle="steps-post")
ax.set_title(f"Lines of {name}, code and comments together")
ax.set_ylabel("lines")
ax.set_ylim(bottom=0)

locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

# Few enough series here to sit inside the axes, which keeps the figure
# rectangular for a page rather than paying width for a legend column.
ax.legend()

In [ ]:
# Uncomment to save a PDF for publications.
# ax.set_title("")
# fig.savefig(f"{name}-combined.pdf", bbox_inches="tight")